# Milestone 2 — Data Parsing and Standardization

- **Course:** CSC14119 — Introduction to Data Science
- **Group:** 03
- **Project:** arXiv Project
- **Report site:** [GitHub Pages](https://duynhat1705.github.io/Intro2DS-Group3-Report/)
- **Repositories:** [Report](https://github.com/DuyNhat1705/Intro2DS-Group3-Report) | [Source code](https://github.com/DuyNhat1705/Intro2DS-Group3-Source)

# Setup and Reporting Data

- Specify the Python environment, package versions, parser revision, and reporting configuration.
- Locate the Milestone 1 publication/version inventory, processed outputs, parser logs, and benchmark records.
- Add executable imports and data-loading cells once the Milestone 2 reporting files are available.
- Validate identifiers, version records, and required columns before calculating statistics.
- Generate tables and figures from the actual processed data during book compilation. Keep missing measurements distinct from zero.

**Planned output:** A generated table describing the reporting inputs, their scope, and their provenance.


# 1. Overview and Scope

## 1.1 Objectives
Explain how raw LaTeX sources become a cleaned hierarchical store and a unified reference collection for Milestone 3.

## 1.2 Input Corpus and Version Coverage
Describe the publications collected in Milestone 1 and all versions expected for each publication. Every publication and version remains in the inventory; versions without TeX sources are recorded as empty and yield no parsed elements.

## 1.3 Pipeline Overview
Describe the stages: source inventory → root detection and multi-file gathering → hierarchy construction → cleanup and math normalization → reference extraction and deduplication → citation rewriting → final content deduplication and export. Explain the actual ordering used by the implementation.

**Planned output:** A pipeline diagram and a generated corpus summary table.


## 2. Hierarchical Parsing

### 2.1 Publication Root and Multi-file Gathering
- Explain how the root containing `\documentclass` is identified, including ambiguous or missing roots.
- Follow `\input` and `\include` transitively, preserving document order and resolving relative paths.
- Ignore commented-out include commands and exclude files unreachable from the root.
- Document handling of missing included files, recursive inclusion, and unsupported commands.

### 2.2 Hierarchy Construction
Describe the document root, chapters/sections, subsections, paragraphs, and other logical divisions. Explain how levels are inferred from each publication's structure, including skipped levels and unnumbered headings.

### 2.3 Leaf Elements and Lists
- Use sentences, block formulas, and figures as leaves; classify tables as figures.
- Describe sentence boundaries without splitting inside mathematics, commands, abbreviations such as `i.e.`, `et al.`, or `Fig. 1`, URLs, or decimals.
- Represent itemized lists as branching components, with each item at the next level and sentence leaves beneath it when appropriate.

### 2.4 Inclusion and Exclusion Rules
Exclude reference sections from the hierarchy while extracting their references separately. Include acknowledgements and appendices, including starred headings such as `\section*`.

**Planned outputs:** A representative multi-file include graph, an original-LaTeX-to-tree example, and a generated element-count table by kind.


## 3. Cleanup and Standardization

### 3.1 Semantic Cleanup
Explain how formatting-only commands and options, such as `\centering`, `[htpb]`, and `\midrule`, are removed while preserving meaningful text and structure. Describe whitespace normalization and handling of comments and escaped characters.

### 3.2 Mathematical Normalization
Describe conversion of inline mathematics to one format, such as `$...$`, and block mathematics to one format, such as `\begin{equation}...\end{equation}`. Explain handling of the math environments actually encountered.

### 3.3 Consistent Normalization
Explain when cleanup occurs relative to sentence splitting, citation rewriting, and deduplication. Ensure exact content matching uses the final cleaned representation.

**Planned output:** A generated before/after example table covering prose, formatting, inline math, block math, and tables.


## 4. Reference Extraction and Cross-version Deduplication

### 4.1 Reference Extraction

The reference extraction pipeline ingests bibliographic records from raw LaTeX sources and normalizes heterogeneous citation declarations into standardized, schema-compliant `BibtexEntry` structures. Citations in scientific preprints exhibit substantial structural variability, ranging from well-formed external BibTeX databases (`.bib`) to unstructured inline `\bibitem` environments embedded within the manuscript body. To guarantee completeness and high-fidelity parsing across all publication versions, the extraction module implements a tiered ingestion architecture coupled with deterministic regular expression parsing, automated API enrichment, and LLM-assisted fallback parsing.

#### Ingestion Hierarchy and Source Priority
The parser resolves bibliography sources for each version according to a strict priority hierarchy:
1. **Dedicated BibTeX Databases (`.bib`):** The version directory is recursively searched for standalone `.bib` files. When detected, entries are parsed using `bibtexparser`, directly mapping native BibTeX attributes (`ENTRYTYPE`, `ID`, `title`, `author`, `year`, `doi`, `journal`, `url`) to standardized fields. Entries parsed via this channel represent ground-truth authorial definitions and are assigned a maximum baseline confidence score of $1.0$ (`source_method = "bib_file"`).
2. **Inline `\bibitem` Blocks (`.tex`):** When external `.bib` files are absent, the parser traverses all `.tex` files along the resolved compilation path (transitively following `\input` and `\include` directives). The bibliography environment `\begin{thebibliography} ... \end{thebibliography}` is located, and individual citations are segmented using the regex:
   ```regex
   \bibitem(?:\[[^\]]*\])?\{([^}]+)\}\s*(.*?)(?=\bibitem|\end\{thebibliography\})
   ```

#### Regular Expression Parsing Engine
For inline `\bibitem` blocks, a modular regular expression engine extracts constituent metadata attributes from unstructured text:
* **Title and Author Delimitation:** Authors typically precede the work's title, while publication venues succeed it. The parser establishes title boundaries by detecting LaTeX quotation marks (`` ``...'' `` or `"..."`) or typographic emphasis commands (`\textit{}`, `\emph{}`, `\textbf{}`). The preceding string is isolated as the author segment, stripped of LaTeX formatting macros (e.g., `\~`, curly braces), normalized across coordination forms (`and`, `et al.`), and tokenized into individual author entities. When explicit quotation or font markup is absent, a period-delimited heuristic decomposes the block into author, title, and venue segments.
* **Temporal and Identifier Attributes:** Publication years are matched using bounded four-digit patterns ($19xx$--$20xx$). Digital Object Identifiers (DOIs) are resolved via standard prefix matching (`10.\d{4,9}/[-._;()/:A-Za-z0-9]+`), while web URLs are extracted and cleansed of trailing punctuation.
* **Downstream Link Anchors (arXiv ID Hints):** To establish deterministic linkage for the reference matching pipeline in Milestone 3, the engine scans reference text and hyperlinks for arXiv identifier patterns (`(?:arxiv(?:\.org)?(?:/(?:abs|pdf))?[:\/\s]+)(\d{4}\.\d{4,5}(?:v\d+)?)`). Identified IDs are recorded in the `arxiv_id_hint` field, providing high-confidence priors for inter-paper citation networks.
* **Entry Type Heuristics:** References are categorized into canonical BibTeX types (`inproceedings`, `article`, `phdthesis`, `book`, `misc`) using venue keyword heuristics (e.g., *proceedings*, *conference*, *symposium*, *transactions*, *dissertation*).

#### Automated CrossRef API Auto-Enrichment
Unstructured inline bibliographies frequently omit key metadata, such as DOIs, complete author rosters, or publication years. When candidate entries lack these vital fields but retain an identifiable title, the pipeline initiates automated enrichment via the CrossRef REST Open API (`api.crossref.org/works`). Queries are dispatched with descriptive user-agent headers adhering to Open API etiquette. The returned JSON payload supplies verified publisher metadata:
* Title normalization (correcting LaTeX casing anomalies and typos).
* Resolution of complete author rosters (given and family names).
* Imputation of canonical publication years, container venue titles, and persistent DOIs.

Enriched records transition to `source_method = "crossref"`, substantially enhancing corpus completeness.

#### Robustness and LLM Fallback Mechanism
For severely malformed LaTeX snippets—such as deeply nested user-defined TeX macros, broken delimiters, or non-standard multilingual citation conventions where deterministic regex matching fails to recover both title and author—the module activates an intelligent fallback channel. The raw citation block is passed to a lightweight instruction-tuned language model (`gemini-1.5-flash`) with structured JSON schema decoding (`response_mime_type = "application/json"`). This ensures robust zero-loss parsing even on non-standard and legacy notations (`source_method = "llm"`).

#### Confidence Scoring Model
To quantify the reliability of extracted entries prior to downstream deduplication and database commitment, a deterministic confidence score $C \in [0.0, 1.0]$ is computed:
$$C = \min\left(1.0, \; 0.45 \cdot \mathbb{I}_{\text{title}} + 0.25 \cdot \mathbb{I}_{\text{authors}} + 0.15 \cdot \mathbb{I}_{\text{year}} + 0.15 \cdot (\mathbb{I}_{\text{doi}} \lor \mathbb{I}_{\text{arxiv\_hint}})\right)$$
where $\mathbb{I}_{x}$ denotes an indicator function verifying the presence of non-empty, validated field values. Entries achieving $C \ge 0.70$ represent publication-grade records containing both core bibliographic attributes and persistent identifiers.

#### Illustrative Transformation Example
The following transformation illustrates the end-to-end normalization of an unstructured inline `\bibitem` into a unified `BibtexEntry`:

**Raw Input (`\bibitem`):**
```latex
\bibitem[Vaswani et al.(2017)]{vaswani2017attention}
A.~Vaswani, N.~Shazeer, N.~Parmar, et~al., ``Attention is all you need,''
in \textit{Advances in Neural Information Processing Systems}, vol.~30, 2017.
arXiv: 1706.03762.
```

**Standardized Output (`BibtexEntry`):**
```json
{
  "arxiv_id": "2310.00001",
  "citation_key": "vaswani2017attention",
  "entry_type": "inproceedings",
  "title": "Attention is all you need",
  "authors": ["A. Vaswani", "N. Shazeer", "N. Parmar"],
  "year": 2017,
  "journal": "Advances in Neural Information Processing Systems",
  "doi": null,
  "url": null,
  "arxiv_id_hint": "1706.03762",
  "source_method": "regex",
  "confidence_score": 1.0
}
```

### 4.2 Duplicate Identification and Canonical Keys
Explain how references to the same underlying work are identified within a version and across all versions of one publication. Document the matching rules and deterministic canonical-key selection.

### 4.3 Field Union and Citation Rewriting
- Merge the fields of duplicate entries so information supplied in earlier or later versions is retained.
- State how conflicting nonempty field values are resolved and audited.
- Rewrite affected `\cite{}` keys to the canonical key, including citations containing multiple keys.
- Check that rewritten citations resolve to the unified bibliography; report unresolved keys.

**Planned outputs:** A worked cross-version example showing different keys, complementary fields, one merged entry, and rewritten citations; generated counts of references before and after deduplication.

## 5. Shared Elements and Output Organization

### 5.1 Full-text Deduplication Across Versions
Explain exact matching after cleanup for both leaves and components. Identical cleaned content across versions of a publication shares one `element_id`; each version retains its own parent relations. Describe element-kind compatibility and repeated occurrences in a version.

### 5.2 Identifiers and Version-specific Hierarchy
Describe publication-prefixed identifiers with discriminants unique within the publication. Illustrate a shared sentence linked to different parents in two versions. Explain document-root representation and preservation of empty-version records.

### 5.3 Output Tables
Describe the actual storage format and how it supports the three required tables.

| Table | Required columns | Purpose |
|---|---|---|
| `elements` | `element_id`, `arxiv_id`, `kind`, `content` | One row per distinct cleaned element across a publication's versions |
| `hierarchy` | `arxiv_id`, `version`, `element_id`, `parent_id` | One row per node-parent edge in each version |
| `bibtex` | `arxiv_id`, `citation_key`, `entry_type`, `title`, `authors`, `year`, `doi`, `journal`, `url` | Unified reference entries after merging |

Document string/non-null identifier fields, integer versions, controlled vocabularies for `kind` and `entry_type`, author lists, and nullable integer years. Explain where the complete version inventory is kept, since empty versions produce no hierarchy edges.

**Planned outputs:** Generated sample rows from all three tables and a cross-version shared-element example.


## 6. Data Statistics and Coverage

### 6.1 Parsing Success and Failure
Define successful parsing and the denominators used. Report both success among versions with TeX sources and end-to-end coverage among all expected versions. Separate successful, failed, and empty/no-TeX outcomes.

### 6.2 Version Coverage for Every Publication
Generate a complete per-publication table with expected versions, versions with TeX, successfully parsed versions, failed versions, empty versions, and coverage. Use compact plots to summarize this table without replacing the full inventory.

### 6.3 Hierarchy and Element Statistics
Report element counts by kind, hierarchy depth, and elements per publication/version. Define how counts distinguish per-version occurrences from distinct stored elements.

### 6.4 Deduplication Results
Report content and reference counts before and after deduplication within each publication's version set, plus corpus totals. Define the reduction rate as `1 - after / before`, handling a zero denominator explicitly. Separate any within-version reference duplicates from cross-version merging.

### 6.5 Interpretation
Explain unusual distributions, version differences, failure categories, and the effect of cleanup and deduplication.

**Planned outputs:** Success-status chart; complete publication/version coverage table; element-kind and depth plots; before/after content and reference count tables and charts. Generate all outputs from reporting data during compilation.


## 7. Correctness Checks and Error Analysis

### 7.1 Structural and Data Integrity
Check for dangling element/parent IDs, cycles, invalid version links, duplicate identifiers, and schema/type violations. Verify inclusion of acknowledgements/appendices and exclusion of reference sections from the hierarchy.

### 7.2 Targeted Parser Examples
Document representative checks for commented includes, nested input files, sentence-boundary exceptions, math environments, nested lists, and tables as figures.

### 7.3 Deduplication and Citation Integrity
Verify that identical cleaned content across versions reuses identifiers, each version keeps the correct parent relations, merged references retain complementary fields, and obsolete citation keys are rewritten.

### 7.4 Failure Analysis and Limitations
Describe the inspection method and examples of failure or partial parsing. Quantify affected cases where logs permit, and discuss consequences for downstream use.

**Planned outputs:** Generated validation summary and representative source/output/error examples. Distinguish manually inspected evidence from corpus-wide checks.


## 8. Performance Evaluation

### 8.1 Benchmark Environment and Method
Run the final benchmark on a **Google Colab CPU-only instance**. Record CPU, available RAM, disk capacity, Python/package versions, corpus size, parser configuration, concurrency, timing boundaries, and monitoring method.

### 8.2 Running Time
Report end-to-end parser wall time and average time per publication, with a clear denominator. Explain whether loading, normalization, deduplication, and writing outputs are included. Use per-publication distributions and stage timings when available.

### 8.3 Memory Footprint and Storage
Report maximum RAM used and final output storage size. State whether monitoring covers child processes and how frequently memory is sampled. Also measure peak disk usage against the Colab instance's capacity; distinguish it from final output size.

### 8.4 Interpretation
Discuss bottlenecks and the relationship between source size, number of versions, element counts, time, and memory. State measurement limitations.

**Planned outputs:** Generated benchmark summary table, processing-time distribution, RAM-over-time plot, and output-size breakdown. No runtime upper limit is imposed; disk use must fit the Colab instance.


# 9. Conclusion

## 9.1 Key Results & Findings

---

## 9.2 Limitations & Challenges

